# Calculating semantic tag statistics for lemmas

This notebook does two things:
1. **Calculates how much of the Estonian reference Corpus the aggregated ekilex tags cover**
2. **Creates a new database table with lemmas, their tag and frequency to see if the most frequent lemmas recieved an aggregated ekilex tag automatically or need further (possibly manual) tagging**

## Tag types
There are 5 tags: location, time, event, state, not_location. These have been combined from various ekilex semantic_type tags:

* **location** (words that are locations with very high likelyhood): *'koht', 'koht_ala', 'koht_asutus', 'koht_geogr', 'koht_geogr_maailmajagu', 'koht_geogr_veekogu', 'koht_hoone', 'koht_kehaosa', 'koht_loodus', 'koht_suund/asend', 'abstr_asend/suund', 'ese_anum', 'omadus_koht'*
    * If a word had several semantic types, then the word was considered a location if one tag was from the list above and the others belonged to the
      semantic types of *ese_instru, ese, ese_kunst, ese_raha, ese_semio, ese_riie, taim, objekt_loodus, objekt, osa, nähtus_loodus*
* **time**: *'aeg', 'aeg_aastaaeg', 'aeg_kuu', 'aeg_nädalapäev', 'aeg_tähtpäev'*
    * If a word had several semantic types, then the word was considered a time if one tag was from the list above and the others belonged to the
      semantic types of *esitus, nähtus_loodus, omadus_aeg*
* **state**: *'seisund', 'seisund_haigus', 'seisund_füüs'*
    * If a word had several semantic types, then the word was considered a time if one tag was from the list above and the others belonged to the
      semantic types of *nähtus_psühh, nähtus, nähtus_loodus, omadus_psühh, abstr_asend/suund, abstr_konkr_omadus, ese_raha*
* **event**: *sündmus*
    * If a word had several semantic types, then the word was considered a time if one tag was from the list above and the others belonged to the
      semantic types of *tegevus, tegevus_tegu, ese_kunst, abstr/konkr, nähtus, toit, nähtus_füüs, tegevus_kõnetegu, tegevus_mäng*
* **not_location**: (words that can't be locations or can be locations very very rarely (such as people)): *ese_raha, ese_riie, esitus_arv, esitus_keel, esitus_keel_suhtlus, esitus_keel_täht, esitus_tiitel, esitus_tähis, in_elukutse, in_müt, in_omadus, in_rahvas, in_roll, in_tegija, amet, konkr_omadus, käsklus, loom_liik, loom_omadus, loom_putukas, nähtus_psühh, omadus, omadus_abstr, omadus_aeg, omadus_füüs, omadus_kval, omadus_psühh, tegevus_muutus, tegevus_tegu, omadus_füüs_värv*

In [1]:
import sqlite3

In [2]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

## Ekilex tag coverage

In [3]:
# ### How many words in spatial cases the reference corpus has
cursor.execute("SELECT lemma FROM spatial_obl")
lemmas = cursor.fetchall()

cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag IS NOT NULL")
not_empty = cursor.fetchall()
coverage = (not_empty[0][0]/len(lemmas))*100
print("Ekilex tags cover " + str(round(coverage, 2)) + "% of words in spatial cases in the Estonian Reference corpus")

cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'location'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("\nPercentages")
print("Location: " + str(round(coverage, 2)) + "% ")

cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'time'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("Time: " + str(round(coverage, 2)) + "% ")

cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'event'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("Event: " + str(round(coverage, 2)) + "% ")

cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'state'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("State: " + str(round(coverage, 2)) + "% ")

cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'not_location'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("Not_location: " + str(round(coverage, 2)) + "% ")

Ekilex tags cover 28.76% of words in spatial cases in the Estonian Reference corpus

Percentages
Location: 13.34% 
Time: 6.22% 
Event: 2.9% 
State: 0.84% 
Not_location: 5.45% 


## Create new lemma+tag+lemma_frequency table in the database
Used to see if the most frequent words are covered by ekilex tags or need to be manually tagged.
Database table has unique lemmas, what ekilex tag they recieved and how many times the lemma appears with a spatial case in the Estonian reference corpus

In [4]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS lemma_frequency_tag")

# Step 1: Create the new results table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS lemma_frequency_tag (
        lemma TEXT,
        tag TEXT,
        lemma_count INT
    )
""")

# Step 2: Aggregate counts and calculate percentages
cursor.execute("""
    INSERT INTO lemma_frequency_tag (lemma, tag, lemma_count)
    SELECT 
        lemma, 
        ekilex_tag, 
        COUNT(*) AS lemma_count
    FROM spatial_obl
    GROUP BY lemma
""")

# Commit and close
conn.commit()
conn.close()